# Peformance comparison of different sparse data formats 

In this notebook we analyze the performance of various algorithms, including MLPs, dense dynamical systems, and various sorts of sparse dynamical systems.

In [16]:
import torch
import warnings
import time
warnings.filterwarnings("ignore")
from iterativennsimple.MaskedLinear import MaskedLinear

import pandas as pd
import plotly.express as px
import numpy as np

In [17]:
results = {}

In [18]:
def generate(size, entries, device="cuda"):
    """create a variety of sparse matrices 

    Args:
        size (int, optional): Size of the square matrix. Defaults to 1000.
        entries (int, optional): Total number of non-zero entries. Note this is an upper bound, but should be close to the actual size. Defaults to 23*1000.
        device (str, optional): "cuda" or "cpu". Defaults to "cuda".

    Returns:
        dict: Dictionary of the sparse matrices
    """ 
    output = {}
    # We first create a COO tensor since that is easier to create
    indices = torch.randint(0, size, (entries,2))
    vals = torch.randn(entries)
    coo = torch.sparse_coo_tensor(indices.t(), vals, (size, size), device=device)
    coo = coo.coalesce()

    # Then we convert it to CSC, CSR and dense
    dense = coo.to_dense()
    csc = coo.to_sparse_csc()
    csr = coo.to_sparse_csr()
    
    # We also create the MaskedLinear and SparseLinear objects
    class moduleWrapper(object):
        def __init__(self, module, device=device):
            self.module = module
            self.device = device
        def __matmul__(self, x):
            return self.module(x.T).T
        
    maskedLinear = moduleWrapper(MaskedLinear.from_coo(coo).to(device), device)

    output["dense"] = dense
    output["coo"] = coo
    output["csc"] = csc
    output["csr"] = csr
    output["maskedLinear"] = maskedLinear
    return output

In [19]:
## check that all of the matrices are the same operator
def matrix_check(size, entries, device):
    # Generate the matrices
    matrices = generate(size, entries, device=device)
    # Size of the RHS
    x_cols = 100
    x = torch.randn(size, x_cols, device=device)

    print('Matrix-matrix multiplication check')
    y_true = None
    base_name = None

    for matrix, matrix_name in zip(matrices.values(), matrices.keys()):
        y = matrix @ x
        if y_true is None:
            y_true = y
            base_name = matrix_name
        else:
            # print the frobenius norm of the difference
            print(f"||{matrix_name} - {base_name}||_F: {torch.norm(y-y_true)}")
            

In [20]:
matrix_check(1000, 23*1000, "cpu")

Matrix-matrix multiplication check
||coo - dense||_F: 0.00015572998381685466
||csc - dense||_F: 0.0001628838654141873
||csr - dense||_F: 0.0001628838654141873
||maskedLinear - dense||_F: 3.270631714258343e-05


In [21]:
matrix_check(1000, 23*1000, "cuda")

Matrix-matrix multiplication check
||coo - dense||_F: 0.00014532080967910588
||csc - dense||_F: 0.00014536465459968895
||csr - dense||_F: 0.0001453307195333764
||maskedLinear - dense||_F: 0.00013229680189397186


In [22]:
def run_timing(size, entries, device, syncgpu):
    # test if cuda is available
    if device == "cuda":
        if not torch.cuda.is_available():
            print("CUDA is not available, using CPU instead")
            device = "cpu"

    print(f"Running on {device}")
    print(f"Size: {size}")
    print(f"Entries: {entries}")
    print(f"Synchronize GPU: {syncgpu}")
    
    # Generate the matrices
    matrices = generate(size, entries, device=device)

    # Size of the RHS
    x_cols = 100

    # Compute the timings
    print('Matrix-matrix multiplication timings:')
    base_time = None
    base_name = None
    all_times = {}
    for matrix, matrix_name in zip(matrices.values(), matrices.keys()):
        # first, do a few runs to warm up the cache
        for i in range(2):
            x = torch.randn(size, x_cols, device=device)
            y = matrix @ x
        # now do the timings
        matrix_times = []
        for i in range(5):
            x = torch.randn(size, x_cols, device=device)    
            if syncgpu:
                torch.cuda.synchronize()
            start = time.perf_counter()
            y = matrix @ x
            if syncgpu:
                torch.cuda.synchronize()
            matrix_time = time.perf_counter()-start
            matrix_times.append(matrix_time)
        avg_matrix_time = sum(matrix_times)/len(matrix_times)
        min_matrix_time = min(matrix_times)
        max_matrix_time = max(matrix_times)
        if base_time is None:
            base_time = avg_matrix_time
            base_name = matrix_name

        name = device+'_'+matrix_name
        all_times[name] = avg_matrix_time

        print(f"{name} avg time:", avg_matrix_time)
        print(f"{name} min time:", min_matrix_time)
        print(f"{name} max time:", max_matrix_time)
        print(f"{name} speedup over {base_name}:", base_time/avg_matrix_time)


    results.update(all_times)
    return all_times

In [23]:
matrix_size = 1024
matrix_entries = 103*matrix_size
# matrix_size = 25000
# matrix_entries = 250*matrix_size
print("matrix size:", matrix_size)
print("Total number of entries:", matrix_size**2)
print("Active entries:", matrix_entries)
print("Ratio of active entries to total entries:", matrix_entries/(matrix_size**2))


matrix size: 1024
Total number of entries: 1048576
Active entries: 105472
Ratio of active entries to total entries: 0.1005859375


In [24]:
all_times = run_timing(matrix_size, matrix_entries, "cpu", True)

Running on cpu
Size: 1024
Entries: 105472
Synchronize GPU: True
Matrix-matrix multiplication timings:
cpu_dense avg time: 0.0003684077993966639
cpu_dense min time: 0.00035750301321968436
cpu_dense max time: 0.0003794939839281142
cpu_dense speedup over dense: 1.0
cpu_coo avg time: 0.003355407784692943
cpu_coo min time: 0.003122970985714346
cpu_coo max time: 0.004171526990830898
cpu_coo speedup over dense: 0.1097952389206778
cpu_csc avg time: 0.0022134187980554997
cpu_csc min time: 0.0020282199839130044
cpu_csc max time: 0.0023473610053770244
cpu_csc speedup over dense: 0.16644288000097954
cpu_csr avg time: 0.00018478621495887637
cpu_csr min time: 0.0001456339959986508
cpu_csr max time: 0.00020799203775823116
cpu_csr speedup over dense: 1.9936974166533579
cpu_maskedLinear avg time: 0.0005718687898479402
cpu_maskedLinear min time: 0.0005473410128615797
cpu_maskedLinear max time: 0.0006228729616850615
cpu_maskedLinear speedup over dense: 0.6442173553388417


In [25]:
# Print the amount of free memory in GB
if torch.cuda.is_available():
    free_memory = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)
    print(f"Free memory on GPU: {free_memory / (1024**3):.2f} GB")

Free memory on GPU: 23.50 GB


In [26]:
all_times = run_timing(matrix_size, matrix_entries, "cuda", True)

Running on cuda
Size: 1024
Entries: 105472
Synchronize GPU: True
Matrix-matrix multiplication timings:
cuda_dense avg time: 3.041560994461179e-05
cuda_dense min time: 2.9295042622834444e-05
cuda_dense max time: 3.331300104036927e-05
cuda_dense speedup over dense: 1.0
cuda_coo avg time: 9.511120151728392e-05
cuda_coo min time: 9.198300540447235e-05
cuda_coo max time: 0.00010293396189808846
cuda_coo speedup over dense: 0.31978998750304466
cuda_csc avg time: 0.00031734359217807653
cuda_csc min time: 0.0002904169959947467
cuda_csc max time: 0.0003384369774721563
cuda_csc speedup over dense: 0.09584441184350163
cuda_csr avg time: 3.6043825093656776e-05
cuda_csr min time: 3.546703374013305e-05
cuda_csr max time: 3.7100049667060375e-05
cuda_csr speedup over dense: 0.8438507806976492
cuda_maskedLinear avg time: 7.80288246460259e-05
cuda_maskedLinear min time: 7.582304533571005e-05
cuda_maskedLinear max time: 8.341699140146375e-05
cuda_maskedLinear speedup over dense: 0.38979966804050653


In [27]:
# Create a dataframe with row and column names for the heatmap
df = pd.DataFrame(columns=results.keys(), index=results.keys())

# Fill in df with the values we want to display
for name1, time1 in results.items():
    for name2, time2 in results.items():
        df.loc[name1, name2] = time1/time2
# Create the heatmap and make it large
# Plot the original heatmap
fig = px.imshow(df, text_auto=True, color_continuous_scale='Jet', width=1000, height=900, title="Relative Timing Heatmap")
fig.show()

# Plot the log10 heatmap
df_log = np.log10(df.astype(float))
fig_log = px.imshow(df_log, text_auto=True, color_continuous_scale='Jet', width=1000, height=900, title="Log10 Relative Timing Heatmap")
fig_log.show()

In [28]:
def train_least_squares(size, entries, device, syncgpu, epochs=30):
    # test if cuda is available
    if device == "cuda":
        if not torch.cuda.is_available():
            print("CUDA is not available, using CPU instead")
            device = "cpu"

    print(f"Running on {device}")
    print(f"Size: {size}")
    print(f"Entries: {entries}")
    print(f"Synchronize GPU: {syncgpu}")
    
    # Generate the matrices
    matrices = generate(size, entries, device=device)

    # Generate the data
    X = torch.randn(10, size, device=device)  # Example input data
    Y = torch.randn(10, size, device=device)  # Example target data

    # Define the loss function
    criterion = torch.nn.MSELoss()

    # Wrap a model around the sparse matrix
    class matrixWrapper(torch.nn.Module):
        def __init__(self, matrix, type="dense"):
            super(matrixWrapper, self).__init__()
            self.type = type
            self.matrix = torch.nn.Parameter(matrix)
        def forward(self, x):
            if self.type == "dense":
                return torch.mm(x, self.matrix.T)
            else:
                return torch.sparse.mm(self.matrix, x.T).T

    # Train the models
    print('Training least squares models:')
    for matrix, matrix_name in zip(matrices.values(), matrices.keys()):
        if matrix_name in ["coo", "csr"]:
            sparse_model = matrixWrapper(matrix, type="sparse").to(device)
        elif matrix_name in ["dense"]:
            sparse_model = matrixWrapper(matrix, type="dense").to(device)
        elif matrix_name in ["maskedLinear", "sparseLinear"]:
            sparse_model = matrix.module.to(device)
        else:
            continue

        # Define the optimizer
        optimizer = torch.optim.SGD(sparse_model.parameters(), lr=0.01)

        # Training loop
        print(f'-------------{matrix_name} training:--------------')
        for epoch in range(epochs):
            optimizer.zero_grad()
            output = sparse_model(X)
            loss = criterion(output, Y)
            if epoch % 10 == 0:
                print(f'{matrix_name} Epoch {epoch}, Loss: {loss.item()}')
            loss.backward()
            optimizer.step()

        print(f'{matrix_name} final loss: {loss.item()}')
    return matrices

# Example usage
matrices = train_least_squares(size=100, entries=500, device='cuda', syncgpu=True)

Running on cuda
Size: 100
Entries: 500
Synchronize GPU: True
Training least squares models:
-------------dense training:--------------
dense Epoch 0, Loss: 5.134312629699707
dense Epoch 10, Loss: 4.920623302459717
dense Epoch 20, Loss: 4.716353416442871
dense final loss: 4.540205478668213
-------------coo training:--------------
coo Epoch 0, Loss: 5.134312629699707
coo Epoch 10, Loss: 5.107922077178955
coo Epoch 20, Loss: 5.081737518310547
coo final loss: 5.058345794677734
-------------csr training:--------------
csr Epoch 0, Loss: 5.134312629699707
csr Epoch 10, Loss: 5.107922077178955
csr Epoch 20, Loss: 5.081737518310547
csr final loss: 5.058345794677734
-------------maskedLinear training:--------------
maskedLinear Epoch 0, Loss: 5.134312629699707
maskedLinear Epoch 10, Loss: 5.107922077178955
maskedLinear Epoch 20, Loss: 5.081737518310547
maskedLinear final loss: 5.058345794677734
